step 1: convert dot annotation to polygon annotation and create a yaml file

*the annotations are point annotations but for detection, we need bbox*

In [1]:
# install pathology-whole-slide-data
!pip3 install git+https://github.com/DIAGNijmegen/pathology-whole-slide-data@main

  Cloning https://github.com/DIAGNijmegen/pathology-whole-slide-data (to revision main) to /tmp/pip-req-build-qnxfs1we
  Running command git clone -q https://github.com/DIAGNijmegen/pathology-whole-slide-data /tmp/pip-req-build-qnxfs1we
  Using cached concurrentbuffer-0.0.8-py3-none-any.whl (14 kB)
  Using cached dicfg-0.0.13-py3-none-any.whl (9.0 kB)
  Using cached sourcelib-0.0.4-py3-none-any.whl (7.6 kB)
  Using cached numpy-1.24.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Using cached Rtree-1.3.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (543 kB)
  Using cached matplotlib-3.7.5-cp38-cp38-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (9.2 MB)
     |████████████████████████████████| 4.7 MB 13.1 MB/s eta 0:00:01
  Using cached contourpy-1.1.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (301 kB)
  Using cached numpy-1.22.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.9 MB)
  Created wheel for wholeslidedata: filenam

In [2]:
import os
from utils.dot2polygon import dot2polygon
from utils.annot2inf_cell import annot2inf_cell
import glob
import yaml

In [9]:
# make a folder for converted annotations
output_root = r'./data/annotations_polygon'
if not(os.path.isdir(output_root)): os.mkdir (output_root) 

In [ ]:
# convert dot annotations to polygon, in a folder
annotation_dir = r'./data/annotations'
annotation_polygon_dir =  r'./data/annotations_polygon'
if not(os.path.isdir(annotation_polygon_dir)): os.mkdir (annotation_polygon_dir) 
annotation_list = glob.glob(os.path.join(annotation_dir,'*.xml'))
for xml_path in annotation_list:
    print(xml_path)
    output_path = os.path.join(annotation_polygon_dir,os.path.splitext(os.path.basename(xml_path))[0] + '_polygon' + os.path.splitext(os.path.basename(xml_path))[1])
    print(output_path)
    lymphocyte_half_box_size = 4 # the size of half of the bbox around the lymphocyte dot in um
    monocytes_half_box_size = 5 # the size of half of the bbox around the monocytes dot in um
    min_spacing = 0.25
    dot2polygon(xml_path, lymphocyte_half_box_size, monocytes_half_box_size, min_spacing, output_path)

*make a config file:*

In [13]:
def folders_to_yml(wsi_dir: str, 
                   wsa_dir: str,
                   output_dir: str,
                   output_name: str):

    """
    Generate a yaml file to be used as WSD dataconfig from a folder of slides and a folder of annotation or mask files.
    Assumes files use the same name for both the slides and masks.
    """
    
    wsa_list = glob.glob(wsa_dir)

    yaml_dict = {'training': []}
    # yaml_dict = {'training': [], 'validation': []}
    for wsa in wsa_list:
        patient_name = os.path.basename(wsa).split(os.path.basename(wsa_dir).split('*')[1])[0] # monocytes
    #     print(patient_name)
#         if os.path.isfile(os.path.join(wsi_dir,patient_name + ".tif")):
#             wsi = os.path.join(wsi_dir, patient_name + ".tif")
        if os.path.isfile(os.path.join(wsi_dir,patient_name + "_PAS_CPG.tif")):
            wsi = os.path.join(wsi_dir, patient_name + "_PAS_CPG.tif")
            print('match found:    ' , patient_name)
            yaml_dict['training'].append(
                    {"wsa": {"path": str(wsa)}, "wsi": {"path": str(wsi)}})

            # # validation if needed
            # yaml_dict['validation'].append(
            #         {"wsa": {"path": str(wsa)}, "wsi": {"path": str(wsi)}})

        else:
            print('no match found:    ' , patient_name)

    # make a folder for output
    if not(os.path.isdir(output_dir)): os.mkdir (output_dir)


    with open(os.path.join(output_dir,output_name), "w") as file:
            yaml.safe_dump(yaml_dict, file)

In [ ]:
wsi_dir = r'./data/images'
wsa_dir = r'./data/annotations_polygon/*_polygon.xml'
output_dir = r'./configs'
output_name = 'training_2class_all.yml'

folders_to_yml(wsi_dir, 
               wsa_dir,
               output_dir,
               output_name)